# Step 1: Fetch Search Results from Multiple Retrieval Endpoints

This notebook fetches search results for the top TK negative intent queries from multiple retrieval endpoints:
- L2 (end-to-end): `http://catalog-search-service-use1.demm.prd.chewy.com:8080/debug/top-results?term=`
- L1 Hybrid: `http://catalog-search-service-use1.demm.prd.chewy.com:8080/debug/top-results?term=` (with + encoding)
- kNN: `http://catalog-search-service-use1.demm.prd.chewy.com:8080/debug/top-knn-results?term=`
- Lexical: `http://catalog-search-service-use1.demm.prd.chewy.com:8080/debug/top-lexical-results?term=`

In [1]:
import pandas as pd
import requests
import json
import os
from urllib.parse import quote_plus, quote
import time
from datetime import datetime
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Configuration

In [2]:
# Configuration
BASE_URL = "http://catalog-search-service-use1.demm.prd.chewy.com:8080"
TOP_N_QUERIES = 2000  # Top 100 negative intent queries by search volume
MAX_WORKERS = 3 # Concurrent requests
REQUEST_TIMEOUT = 30  # seconds
DELAY_BETWEEN_REQUESTS = 0.1  # seconds

# Output directory
OUTPUT_DIR = "./search_results/control"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# API endpoints
ENDPOINTS = {
    'l2': f"{BASE_URL}/debug/top-results",  # L2 end-to-end (% encoding)
    'l1_hybrid': f"{BASE_URL}/debug/top-results",  # L1 hybrid (+ encoding)
    'knn': f"{BASE_URL}/debug/top-knn-results",
    'lexical': f"{BASE_URL}/debug/top-lexical-results"
}

print(f"Output directory: {OUTPUT_DIR}")
print(f"Endpoints configured: {list(ENDPOINTS.keys())}")

Output directory: ./search_results/control
Endpoints configured: ['l2', 'l1_hybrid', 'knn', 'lexical']


## Load Negative Intent Queries

In [3]:
# Load the negative intent candidates
df = pd.read_csv("negative_intent_candidates(raw_data).csv")

print(f"Total queries in dataset: {len(df)}")
print(f"\nDataset columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

Total queries in dataset: 1000

Dataset columns: ['Q', 'CNT', 'TARGET_X']

First 5 rows:


,Q,CNT,TARGET_X
0,grain free dog food,611730,grain
1,no pull dog harnesses,211607,pull
2,grain free dry cat food,197068,grain
3,grain free wet cat food,162972,grain
4,no hide chews,147107,hide


In [4]:
# Sort by search count and take top N
df_sorted = df.sort_values('CNT', ascending=False)
top_queries = df_sorted.head(TOP_N_QUERIES)

print(f"Selected top {TOP_N_QUERIES} queries by search volume:")
print(f"Search count range: {top_queries['CNT'].min():,} - {top_queries['CNT'].max():,}")
print(f"\nSample queries:")
for i, row in top_queries.head(10).iterrows():
    print(f"  '{row['Q']}' ({row['CNT']:,} searches)")

Selected top 2000 queries by search volume:
Search count range: 967 - 611,730

Sample queries:
  'grain free dog food' (611,730 searches)
  'no pull dog harnesses' (211,607 searches)
  'grain free dry cat food' (197,068 searches)
  'grain free wet cat food' (162,972 searches)
  'no hide chews' (147,107 searches)
  'stuffing free dog toys' (142,348 searches)
  'grain free dog treats' (138,329 searches)
  'rawhide free dog bones' (135,517 searches)
  'pork chomps rawhide free' (129,324 searches)
  'chicken free dry dog food' (124,263 searches)


In [5]:
final_queries = top_queries.copy()

## API Request Functions

In [6]:
def make_api_request(url, timeout=REQUEST_TIMEOUT):
    """Make API request - fails fast on any error"""
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()

def fetch_results_for_query(query, endpoint_type, max_results=36):
    """Fetch search results for a single query from a specific endpoint"""
    
    # URL encoding based on endpoint type
    if endpoint_type == 'l2':
        # L2 uses % encoding
        encoded_query = quote(query)
    else:
        # L1, kNN, lexical use + encoding
        encoded_query = quote_plus(query)
    
    url = f"{ENDPOINTS[endpoint_type]}?term={encoded_query}"
    
    logger.info(f"Fetching {endpoint_type} results for: '{query}'")
    
    results = make_api_request(url)
    
    # Extract entries and limit to top 36
    entries = results['entries']  # Will fail if 'entries' key doesn't exist
    if len(entries) > max_results:
        entries = entries[:max_results]
        logger.info(f"Limited {endpoint_type} results to top {max_results} for query: '{query}'")
    
    # Process entries into structured format
    processed_entries = []
    for rank, entry in enumerate(entries, 1):
        doc_id = entry.get('docId')
        sku_name = entry.get('name', '')  # Assuming 'name' field contains SKU name
        part_number = entry.get('partNumber', '')
        
        # Handle score logic
        if endpoint_type in ['l1_hybrid', 'l2']:
            score = 1/(rank + 1)  # Score is 0 for l1_hybrid and l2
        else:
            score = entry.get('score', 0)
        
        processed_entries.append({
            'search_term': query,
            'endpoint_type': endpoint_type,
            'rank': rank,
            'doc_id': doc_id,
            'part_number': part_number,
            'sku_name': sku_name,
            'score': score
        })
    
    return processed_entries

def get_processed_terms_for_endpoint(endpoint_type):
    """Get list of already processed search terms for an endpoint"""
    csv_filename = f"{endpoint_type}_search_results.csv"
    csv_filepath = os.path.join(OUTPUT_DIR, csv_filename)
    
    if os.path.exists(csv_filepath):
        existing_df = pd.read_csv(csv_filepath)
        return set(existing_df['search_term'].unique())
    return set()

def save_results_to_csv(all_results_by_endpoint):
    """Save all results grouped by endpoint type to separate CSV files"""
    saved_files = []
    
    for endpoint_type, results_list in all_results_by_endpoint.items():
        if not results_list:
            continue
            
        # Convert to DataFrame
        df = pd.DataFrame(results_list)
        
        # Sort by search term and rank
        df = df.sort_values(['search_term', 'rank'])
        
        # Save to CSV
        csv_filename = f"{endpoint_type}_search_results.csv"
        csv_filepath = os.path.join(OUTPUT_DIR, csv_filename)
        
        # Append to existing file if it exists
        if os.path.exists(csv_filepath):
            existing_df = pd.read_csv(csv_filepath)
            df = pd.concat([existing_df, df], ignore_index=True)
            df = df.drop_duplicates(subset=['search_term', 'endpoint_type', 'doc_id'], keep='last')
        
        df.to_csv(csv_filepath, index=False)
        saved_files.append(csv_filepath)
        
        logger.info(f"Saved {len(df)} results to {csv_filepath}")
    
    return saved_files

In [7]:
# Test API with single query to verify endpoints are working
test_query = top_queries.iloc[0]['Q']
print(f"Testing API and processing with query: '{test_query}'")
print("=" * 50)

for endpoint_type in ENDPOINTS.keys():
    print(f"\nTesting {endpoint_type} endpoint...")
    
    try:
        processed_results = fetch_results_for_query(test_query, endpoint_type)
        print(f"✅ {endpoint_type}: SUCCESS")
        print(f"   Processed entries: {len(processed_results)}")
        
        if processed_results:
            sample_entry = processed_results[0]
            print(f"   Sample entry: {sample_entry}")
        
    except Exception as e:
        print(f"❌ {endpoint_type}: FAILED - {e}")
    
    print("-" * 30)

print("\n" + "=" * 50)
print("API test complete. Check processed entry format above.")

INFO:__main__:Fetching l2 results for: 'grain free dog food'


Testing API and processing with query: 'grain free dog food'

Testing l2 endpoint...


INFO:__main__:Fetching l1_hybrid results for: 'grain free dog food'


✅ l2: SUCCESS
   Processed entries: 36
   Sample entry: {'search_term': 'grain free dog food', 'endpoint_type': 'l2', 'rank': 1, 'doc_id': 40899, 'part_number': '55842', 'sku_name': 'Taste of the Wild Southwest Canyon Grain-Free Dry Dog Food, 28-lb bag', 'score': 0.5}
------------------------------

Testing l1_hybrid endpoint...


INFO:__main__:Fetching knn results for: 'grain free dog food'


✅ l1_hybrid: SUCCESS
   Processed entries: 36
   Sample entry: {'search_term': 'grain free dog food', 'endpoint_type': 'l1_hybrid', 'rank': 1, 'doc_id': 40899, 'part_number': '55842', 'sku_name': 'Taste of the Wild Southwest Canyon Grain-Free Dry Dog Food, 28-lb bag', 'score': 0.5}
------------------------------

Testing knn endpoint...


INFO:__main__:Limited knn results to top 36 for query: 'grain free dog food'
INFO:__main__:Fetching lexical results for: 'grain free dog food'


✅ knn: SUCCESS
   Processed entries: 36
   Sample entry: {'search_term': 'grain free dog food', 'endpoint_type': 'knn', 'rank': 1, 'doc_id': 227221, 'part_number': '200594', 'sku_name': 'Merrick Grain-Free Real Salmon & Sweet Potato Recipe Dry Dog Food, 10-lb bag', 'score': 1.6340212}
------------------------------

Testing lexical endpoint...


INFO:__main__:Limited lexical results to top 36 for query: 'grain free dog food'


✅ lexical: SUCCESS
   Processed entries: 36
   Sample entry: {'search_term': 'grain free dog food', 'endpoint_type': 'lexical', 'rank': 1, 'doc_id': 279352, 'part_number': '252887', 'sku_name': 'Portland Pet Food Company Pumpkin Biscuits Grain-Free & Gluten-Free Dog Treats, 5-oz bag', 'score': 322.68527}
------------------------------

API test complete. Check processed entry format above.


In [8]:
print(len(processed_results))
processed_results[:3]

36


[{'search_term': 'grain free dog food',
  'endpoint_type': 'lexical',
  'rank': 1,
  'doc_id': 279352,
  'part_number': '252887',
  'sku_name': 'Portland Pet Food Company Pumpkin Biscuits Grain-Free & Gluten-Free Dog Treats, 5-oz bag',
  'score': 322.68527},
 {'search_term': 'grain free dog food',
  'endpoint_type': 'lexical',
  'rank': 2,
  'doc_id': 181320,
  'part_number': '154551',
  'sku_name': 'Taste of the Wild High Prairie Grain-Free Dry Dog Food, 28-lb bag',
  'score': 321.35376},
 {'search_term': 'grain free dog food',
  'endpoint_type': 'lexical',
  'rank': 3,
  'doc_id': 181318,
  'part_number': '154549',
  'sku_name': 'Taste of the Wild Pacific Stream Smoke-Flavored Salmon Grain-Free Dry Dog Food, 28-lb bag',
  'score': 320.97763}]

## Fetch Search Results

In [9]:
def fetch_all_results(queries_df, max_workers=MAX_WORKERS):
    """Fetch results for all queries from all endpoints and save as CSV files"""
    
    # Check for already processed terms per endpoint
    processed_terms_by_endpoint = {}
    for endpoint_type in ENDPOINTS.keys():
        processed_terms_by_endpoint[endpoint_type] = get_processed_terms_for_endpoint(endpoint_type)
        logger.info(f"{endpoint_type}: {len(processed_terms_by_endpoint[endpoint_type])} terms already processed")
    
    # Filter queries that need processing
    queries_to_process = []
    for _, row in queries_df.iterrows():
        query = row['Q']
        for endpoint_type in ENDPOINTS.keys():
            if query not in processed_terms_by_endpoint[endpoint_type]:
                queries_to_process.append((query, endpoint_type))
    
    total_tasks = len(queries_to_process)
    completed_tasks = 0
    failed_tasks = []
    
    # Store results by endpoint type
    all_results_by_endpoint = {endpoint: [] for endpoint in ENDPOINTS.keys()}
    
    print(f"Total tasks to process: {total_tasks} (skipping already processed)")
    
    if total_tasks == 0:
        print("All queries already processed!")
        return 0, []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_task = {}
        
        for query, endpoint_type in queries_to_process:
            future = executor.submit(fetch_results_for_query, query, endpoint_type)
            future_to_task[future] = (query, endpoint_type)
        
        # Process completed tasks
        with tqdm(total=total_tasks, desc="Fetching results") as pbar:
            for future in as_completed(future_to_task):
                query, endpoint_type = future_to_task[future]
                
                try:
                    result = future.result()  # This is now a list of processed entries
                    if result:
                        all_results_by_endpoint[endpoint_type].extend(result)
                        logger.info(f"Processed {len(result)} entries for '{query}' - {endpoint_type}")
                    else:
                        failed_tasks.append((query, endpoint_type))
                        logger.warning(f"No results for: {query} - {endpoint_type}")
                        
                except Exception as e:
                    failed_tasks.append((query, endpoint_type))
                    logger.error(f"Error processing {query} - {endpoint_type}: {e}")
                
                completed_tasks += 1
                pbar.update(1)
                
                # Small delay to avoid overwhelming the server
                time.sleep(DELAY_BETWEEN_REQUESTS)
    
    # Save all results to CSV files
    saved_files = save_results_to_csv(all_results_by_endpoint)
    
    print(f"\nSaved CSV files: {saved_files}")
    return completed_tasks, failed_tasks

In [10]:
# Start fetching results
start_time = time.time()
completed, failed = fetch_all_results(final_queries)
end_time = time.time()

print(f"\n=== RESULTS SUMMARY ===")
print(f"Completed tasks: {completed}")
print(f"Failed tasks: {len(failed)}")
print(f"Total time: {end_time - start_time:.2f} seconds")

if failed:
    print(f"\nFailed tasks:")
    for query, endpoint in failed:
        print(f"  '{query}' - {endpoint}")

# Show CSV files created
csv_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.csv')]
print(f"\nCSV files created: {len(csv_files)}")
for csv_file in csv_files:
    csv_path = os.path.join(OUTPUT_DIR, csv_file)
    df = pd.read_csv(csv_path)
    print(f"  {csv_file}: {len(df)} rows")

INFO:__main__:l2: 0 terms already processed
INFO:__main__:l1_hybrid: 0 terms already processed
INFO:__main__:knn: 0 terms already processed
INFO:__main__:lexical: 0 terms already processed
INFO:__main__:Fetching l2 results for: 'grain free dog food'
INFO:__main__:Fetching l1_hybrid results for: 'grain free dog food'
INFO:__main__:Fetching knn results for: 'grain free dog food'


Total tasks to process: 4000 (skipping already processed)


Fetching results:   0%|          | 0/4000 [00:00<?, ?it/s]INFO:__main__:Limited knn results to top 36 for query: 'grain free dog food'
INFO:__main__:Fetching lexical results for: 'grain free dog food'
INFO:__main__:Processed 36 entries for 'grain free dog food' - knn
Fetching results:   0%|          | 1/4000 [00:00<33:34,  1.99it/s]INFO:__main__:Fetching l2 results for: 'no pull dog harnesses'
INFO:__main__:Processed 36 entries for 'grain free dog food' - l1_hybrid
Fetching results:   0%|          | 2/4000 [00:00<31:56,  2.09it/s]INFO:__main__:Fetching l1_hybrid results for: 'no pull dog harnesses'
INFO:__main__:Processed 36 entries for 'grain free dog food' - l2
Fetching results:   0%|          | 3/4000 [00:01<20:36,  3.23it/s]INFO:__main__:Fetching knn results for: 'no pull dog harnesses'
INFO:__main__:Fetching lexical results for: 'no pull dog harnesses'
INFO:__main__:Processed 36 entries for 'no pull dog harnesses' - l2
Fetching results:   0%|          | 5/4000 [00:02<27:26,  2.43i


Saved CSV files: ['./search_results/control/l2_search_results.csv', './search_results/control/l1_hybrid_search_results.csv', './search_results/control/knn_search_results.csv', './search_results/control/lexical_search_results.csv']

=== RESULTS SUMMARY ===
Completed tasks: 4000
Failed tasks: 5
Total time: 1157.49 seconds

Failed tasks:
  'royal canon dog food' - lexical
  'royal canon cat food' - lexical
  'cat toys without catnip' - lexical
  'merrick real texas beef + sweet potato recipe grain-free chicken-free adult dry dog food' - lexical
  'royal canon cat' - lexical

CSV files created: 4
  l1_hybrid_search_results.csv: 35692 rows
  l2_search_results.csv: 35687 rows
  knn_search_results.csv: 36000 rows
  lexical_search_results.csv: 31421 rows


# no lexical results 
Failed tasks:
  'royal canon dog food' - lexical
  'royal canon cat food' - lexical
  'cat toys without catnip' - lexical
  'merrick real texas beef + sweet potato recipe grain-free chicken-free adult dry dog food' - lexical
  'royal canon cat' - lexical

## Verify Results

In [11]:
# Check what CSV files were created
csv_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.csv')]
print(f"Total CSV files created: {len(csv_files)}")

# Show details for each endpoint CSV
for csv_file in csv_files:
    print(f"\n{csv_file}:")
    csv_path = os.path.join(OUTPUT_DIR, csv_file)
    df = pd.read_csv(csv_path)
    
    print(f"  Total rows: {len(df)}")
    print(f"  Unique search terms: {df['search_term'].nunique()}")
    print(f"  Unique products: {df['doc_id'].nunique()}")
    print(f"  Columns: {list(df.columns)}")
    
    # Show sample rows
    print(f"  Sample rows:")
    print(df.head(3).to_string(index=False))

Total CSV files created: 4

l1_hybrid_search_results.csv:
  Total rows: 35692
  Unique search terms: 1000
  Unique products: 10554
  Columns: ['search_term', 'endpoint_type', 'rank', 'doc_id', 'part_number', 'sku_name', 'score']
  Sample rows:
                                search_term endpoint_type  rank  doc_id  part_number                                                                                              sku_name    score
2 hounds design freedom no pull dog harness     l1_hybrid     1  155034       127945 2 Hounds Design Freedom No Pull Nylon Dog Harness & Leash, Teal, Medium: 22 to 28-in chest, 1-in wide 0.500000
2 hounds design freedom no pull dog harness     l1_hybrid     2  148102       120925                                                           HDP Big Dog No Pull Dog Harness, Red, Large 0.333333
2 hounds design freedom no pull dog harness     l1_hybrid     3 1098406      1098406             PetSafe Easy Walk Comfort Reflective No Pull Dog Harness, Black, Large:

## Create Summary Report

In [12]:
# Create a summary report
csv_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.csv')]

# Collect statistics from CSV files
csv_stats = {}
for csv_file in csv_files:
    csv_path = os.path.join(OUTPUT_DIR, csv_file)
    df = pd.read_csv(csv_path)
    endpoint_type = csv_file.replace('_search_results.csv', '')
    
    csv_stats[endpoint_type] = {
        'total_rows': len(df),
        'unique_search_terms': df['search_term'].nunique(),
        'unique_products': df['doc_id'].nunique(),
        'filename': csv_file
    }

summary = {
    'execution_date': datetime.now().isoformat(),
    'total_queries_processed': len(final_queries),
    'total_endpoints': len(ENDPOINTS),
    'csv_files_created': len(csv_files),
    'csv_statistics': csv_stats,
    'failed_tasks': failed,
    'queries_processed': final_queries['Q'].tolist(),
    'output_directory': OUTPUT_DIR
}

# Save summary
summary_file = os.path.join(OUTPUT_DIR, 'fetch_summary.json')
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary report saved to: {summary_file}")
print(f"\n=== FETCH COMPLETION ===")
print(f"CSV files created: {len(csv_files)}")
print(f"Ready for Step 2: LLM Evaluation")

Summary report saved to: ./search_results/control/fetch_summary.json

=== FETCH COMPLETION ===
CSV files created: 4
Ready for Step 2: LLM Evaluation
